# Digital Signatures Step by Step. From Bitstrings to RSA Signing and Verification
## A mathematical and practical walkthrough in Python

## Learning Goals
By the end of this notebook, you will understand:
- What text looks like as bytes and bitstrings.
- Why we hash data before signing it.
- How mathematical (textbook) RSA generates a signature.
- The difference between textbook RSA and secure, real-world RSA signatures.
- How to use Python's `cryptography` library to sign and verify messages securely.

## Section 1. What problem digital signatures solve
Imagine you send a message over the internet saying: **"Approve payment of ₹500"**.
If an attacker intercepts this, they could change it to **"Approve payment of ₹50000"**.

Without digital signatures, you face several problems:
1. **Authenticity**: How does the bank know this message *really* came from you and not an impostor?
2. **Integrity**: How does the bank know the amount wasn't changed in transit?
3. **Non-repudiation**: What if you claim later that you never sent the message at all?

Digital signatures solve all these. They act like a tamper-proof cryptographic wax seal.

### Wait, isn't this encryption?
No! **Encryption** hides the message so no one can read it. **Digital signatures** can be applied to public (unencrypted) messages. Their goal is *proof of origin and integrity*, not secrecy.

## Section 2. Messages as bytes and bitstrings
Computers do not understand human text. Under the hood, a string is stored as **bytes**. A byte is simply a group of 8 **bits** (1s and 0s).
Because cryptography is just math, we need to convert our text into numbers (bytes) first.
Let's see what a message looks like in bytes and bits.

In [ ]:
# Define our sample message
message_text = "Approve payment of ₹500"

# Convert text to UTF-8 bytes
message_bytes = message_text.encode('utf-8')

print("1. Original text:", message_text)
print("2. As bytes (raw):", message_bytes)
print("3. As numbers (0-255):", list(message_bytes))

# Convert each byte to an 8-bit binary string to see the actual 0s and 1s
message_bits = ' '.join(f"{b:08b}" for b in message_bytes)
print("4. As bits (binary):", message_bits)

## Section 3. Hashing the message
In practice, we do not sign the message directly. Instead, we sign its **hash**.

**What is a hash function?**
A hash function is a mathematical algorithm that takes input of *any* size, and squashes it down to a **fixed-size fingerprint** (called a digest).

Key properties:
1. **Deterministic**: The same message always gives the exact same hash.
2. **Avalanche Effect**: Changing even ONE letter changes the entire hash completely.
3. **One-way function**: You cannot reverse the hash to get the message back. It is a fingerprint, not a zipped compression.

Let's use Python's built-in `hashlib` to create a SHA-256 hash.

In [ ]:
import hashlib

# Original message hash
hasher = hashlib.sha256()
hasher.update(message_bytes)
digest = hasher.digest()

print(f"Original message: '{message_text}'")
print(f"Hash (Hex format): {digest.hex()}")
print(f"Hash length: {len(digest)} bytes ({len(digest) * 8} bits)")

# Now let's change one single digit in our text
altered_text = "Approve payment of ₹900"
altered_bytes = altered_text.encode('utf-8')

hasher2 = hashlib.sha256()
hasher2.update(altered_bytes)
altered_digest = hasher2.digest()

print(f"\nAltered message:  '{altered_text}'")
print(f"New Hash (Hex):    {altered_digest.hex()}")

print("\nNotice how completely different the two hashes are, even though only one digit changed!")

## Section 4. Public key and private key intuition
Digital signatures use **Asymmetric Cryptography**, which relies on a mathematical pair of keys:

1. **The Private Key**: Think of this as your personal stamp. You **keep it secret**. You use it to **sign** messages.
2. **The Public Key**: Think of this as the instructions for verifying your stamp. You **share it with everyone**. Others use it to **verify** your signature.

Because the two keys are mathematically linked, a signature made by the private key can *only* be verified as mathematically valid by the corresponding public key. No one else can forge the signature because they don't have your private key!

## Section 5. Mathematical RSA background
Let's look at the underlying math using the **RSA algorithm**.
RSA relies on the fact that multiplying two large primes is easy, but factoring their product back into primes is incredibly hard.

For this toy example, we will use very small numbers so we can trace the math clearly.

1. Choose two prime numbers, $p$ and $q$.
2. Compute $n = p \times q$. This $n$ will be part of the public key.
3. Compute Euler's Totient function: $\phi(n) = (p-1)(q-1)$.
4. Choose $e$, the public exponent, such that the greatest common divisor $\gcd(e, \phi(n)) = 1$.
5. Compute $d$, the private exponent, such that solving $e \times d \equiv 1 \pmod{\phi(n)}$ holds true.

Your **Public Key** is $(n, e)$.
Your **Private Key** is $d$ (along with $n$).

In [ ]:
import math

# 1. Pick two small primes
p = 61
q = 53
print(f"Primes chosen: p={p}, q={q}")

# 2. Compute n
n = p * q
print(f"Modulus n = {n}")

# 3. Compute phi(n)
phi_n = (p - 1) * (q - 1)
print(f"Euler's Totient phi(n) = {phi_n}")

# 4. Choose public exponent e (must be coprime with phi_n)
e = 17
assert math.gcd(e, phi_n) == 1
print(f"Public exponent e = {e}")
print(f"PUBLIC KEY is (n={n}, e={e})")

# 5. Compute private exponent d (modular multiplicative inverse)
# We need an integer d such that (d * e) % phi_n == 1
# Python 3.8+ has pow(base, exp, mod) which computes modular inverse when exp=-1
d = pow(e, -1, phi_n)
print(f"Private exponent d = {d}")
print(f"PRIVATE KEY is d = {d} (with n={n})")

## Section 6. Turn hash into an integer
RSA math works exclusively on numbers (integers), not raw bytes.
So, before we can do any math on our SHA-256 hash, we must convert those bytes into one giant integer.

In Python, we can do this using `int.from_bytes()`.

In [ ]:
# Let's take the first two bytes of our previous SHA-256 digest just to keep numbers small for our toy example.
# (If we convert the full 32 bytes, the number will be much larger than n=3233, and RSA requires the message integer to be less than n).

# Toy message hash (picking the first 2 bytes of the digest)
toy_hash_bytes = digest[:2]
# Convert bytes to an integer
h_integer = int.from_bytes(toy_hash_bytes, byteorder='big')

print(f"First 2 bytes of the hash: {toy_hash_bytes.hex()}")
print(f"Converted to integer: h = {h_integer}")

# Because our toy n is 3233, we must ensure h < n for textbook RSA to work correctly over the modulus.
if h_integer >= n:
    # Just a hack for our toy math. Real RSA uses huge n (e.g., 2048 bits) 
    # so the hash integer is always smaller than n.
    h_integer = h_integer % n
    print(f"Reduced modulo n: h = {h_integer}")

## Section 7. Toy RSA signing mathematics
How do we actually sign? It's just modular exponentiation!

**Signing (using Private Key `d`)**:
Take the hash integer $h$, raise it to the power of $d$, and take the remainder dividing by $n$.
$$s = h^d \pmod{n}$$

**Verification (using Public Key `e`)**:
Anyone can take your signature $s$, raise it to the power of $e$, and divide by $n$.
$$v = s^e \pmod{n}$$

If the result $v$ matches the original hash $h$, the signature is **valid**! That means it mathematically must have been created by whoever holds $d$.

*Wait, what does "signing with private key" mean? It literally means computing this powerful mathematical formula where the private exponent acts as the secret key.*

In [ ]:
# 1. SIGNING: Sender computes s = (h ^ d) mod n
# We use Python's built-in 3-argument pow() which does modular exponentiation efficiently.
s = pow(h_integer, d, n)
print(f"SIGNATURE s = {s}")

# 2. VERIFICATION: Receiver computes v = (s ^ e) mod n
v = pow(s, e, n)
print(f"VERIFICATION v = {v}")

# 3. CHECK: Does v match our original hash h?
is_valid = (v == h_integer)
print(f"Does the signature match the hash integer ({h_integer})?  -->  {is_valid}")

## Section 8. Why verification detects tampering
What if an attacker alters the message to "Approve payment of ₹900" but leaves the old signature $s$ attached?
The receiver will compute a new hash for the forged message, and then verify the *old signature* against the new hash.

In [ ]:
# Let's simulate the attacker
# New altered hash bytes (we take the first 2 bytes of altered_digest)
altered_toy_bytes = altered_digest[:2]
altered_h_integer = int.from_bytes(altered_toy_bytes, byteorder='big') % n
print(f"Fake message hash integer: {altered_h_integer}")

# The receiver tries to verify the OLD signature (s) against the NEW hash
v_fake = pow(s, e, n)
print(f"Verification output from signature: {v_fake}")

print(f"Does {v_fake} equal {altered_h_integer}?")
if v_fake == altered_h_integer:
    print("WARNING: Signature is valid! (This shouldn't happen!)")
else:
    print("SUCCESS: VERIFICATION FAILED! The math caught the tampering.")

## Section 9. Limitations of textbook RSA
Wait! Is this toy math all there is to digital signatures?
**NO.**

What we just did is called **Textbook RSA**. In the real world, Textbook RSA is **insecure** and should absolutely never be used in production.
Why?
1. Pre-computation: Because the math structure is perfectly deterministic ($E(m_1 \times m_2) = E(m_1) \times E(m_2)$), attackers can combine previously signed valid signatures to forge new ones.
2. Small numbers: Without strict boundaries, malicious actors can exploit the simple math to back-calculate data.

To be safe, real systems use **Padding**. Padding adds randomness, strict formatting boundaries, and robustness to the hash before it is mathematically processed. The most modern, secure standard for this is called **RSA-PSS** (Probabilistic Signature Scheme).

## Section 10. Real digital signatures in Python with secure libraries
Let's generate a secure, real-world Python key pair using the widely-used library: `cryptography`.
We will use a key size of 2048 bits instead of our tiny $n=3233$ (which was $\approx 11$ bits).

*If you are running this locally, you may need to install the library:*

In [ ]:
%pip install cryptography
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

# Generate a secure 2048-bit RSA Private Key
private_key = rsa.generate_private_key(
    public_exponent=65537, # 65537 is the standard safe 'e' exponent in production
    key_size=2048,
)

# Extract the Public Key from the Private Key
public_key = private_key.public_key()

print("Secure RSA Key Pair Generated!")
print(f"Key Size: {private_key.key_size} bits")

# Usually, we serialize the public key into PEM format (the classic 'BEGIN PUBLIC KEY' text block)
# so we can share it easily over email or network.
pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)
val = pem.decode('utf-8')
print("\nPublic Key excerpt:")
# Truncating display so it doesn't crowd the notebook
lines = val.split('\n')
print('\n'.join(lines[:3]) + '\n...\n' + lines[-2])

## Section 11. Real signing using RSA-PSS and SHA-256
Now we will sign the message securely.
Notice that we provide 3 things to the sign method:
1. The raw message bytes (the library will hash it for us).
2. The Padding algorithm (PSS).
3. The Hash algorithm (SHA256).

The formal flow is:
1. Calculate hash of message.
2. Apply PSS padding rules (adds secure randomization).
3. Perform the mathematical operations $s = h^d \pmod{n}$.
4. Output the result bytes.

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding

real_message = b"Approve payment of Rs 500"
print(f"Message to sign: {real_message}")

# 1. Provide padding rules (PSS)
# 2. Provide hash algorithm details (SHA256)
secure_signature = private_key.sign(
    real_message,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print(f"\nSignature created!")
print(f"Signature length: {len(secure_signature)} bytes")
# Print the first 64 hex characters of the signature simply to show what it looks like
print(f"Signature Output (truncated hex): {secure_signature.hex()[:64]}...")

## Section 12. Real verification using public key
To verify, the receiver checks the formula logically.
If it succeeds, Python silently permits the code to continue execution.
If it fails, Python raises a loud `InvalidSignature` exception, preventing you from accepting forged data.

In [ ]:
from cryptography.exceptions import InvalidSignature

print("Scenario A: Verifying the original message with the correct signature")
try:
    public_key.verify(
        secure_signature,
        real_message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("   -> Success! The signature is valid for this message.")
except InvalidSignature:
    print("   -> ERROR: Invalid Signature!")

print("\nScenario B: An attacker alters the message to '...Rs 900'")
hacked_message = b"Approve payment of Rs 900"
try:
    public_key.verify(
        secure_signature,
        hacked_message, # <--- The tampered message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("   -> Success! The signature is valid.") # Should not reach here
except InvalidSignature:
    print("   -> ERROR: Invalid Signature detected! The tampering was caught.")

## Section 13. Connect the toy math to the real implementation
Let's bridge the gap between our toy math and the real-world code:

1. **The Integer Conversion**: Our real `private_key.sign()` implicitly converted the padded bytes to a huge 2048-bit integer behind the scenes.
2. **The Math**: The same abstract formula $s = h^d \pmod{n}$ was executed under the hood in C libraries (like OpenSSL).
3. **The Role of PSS Padding**: Before doing the math, PSS added random "salt" data into the block according to a rigorous Standard. If you sign the exact same message twice with PSS, you will get two totally different 2048-bit signatures. This blocks attackers from building deterministic forgery chains.
4. **Why secure libraries are required**: You cannot simply write `pow(h, d, n)` manually because you will fail to implement the delicate formatting of padding properly, leading to devastating system vulnerabilities.

## Section 14. Mini end-to-end example
Let's wrap it all up into one clear narrative block showing Sender Bob communicating securely with Receiver Alice.

In [ ]:
# --- SENDER (Bob) ---
msg = b"Hello Alice, this is Bob. Transfer $1M to account 1234."
print(f"Bob signs message: {msg}")

# Bob securely signs it using his isolated private key object
signature = private_key.sign(
    msg,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256()
)

# Bob sends msg + signature over the internet...
# --------------------

# --- NETWORK (Attacker intercepts!) ---
print("\nAttacker intercepts and attempts to alter payload...")
msg_received_by_alice = b"Hello Alice, this is Bob. Transfer $1M to account 9999."
print(f"Altered message arrives at Alice: {msg_received_by_alice}")
# --------------------------------------

# --- RECEIVER (Alice) ---
try:
    print("\nAlice attempts to verify...")
    public_key.verify(
        signature,
        msg_received_by_alice,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256()
    )
    print("Alice: VERIFIED!")
except InvalidSignature:
    print("Alice: VERIFICATION FAILED! Someone tampered with this. Discarding message!")

## Section 15. Common misconceptions
- ❌ **"Digital signature encrypts the message with the private key."**
  - ✅ **No**, the message is sent in plain text alongside the signature. Only the *hash* was processed mathematically. The goal is verification, not hiding text.
- ❌ **"The public key can create a signature if you try hard enough."**
  - ✅ **No**, calculating the private key from the public key means factoring the 2048-bit modulus $n$. This is mathematically infeasible to compute in billions of years.
- ❌ **"Hashing compresses my message so I can decompress it later."**
  - ✅ **No**, hashing is a destructive, one-way process. You absolutely cannot recover a file from its hash fingerprint.
- ❌ **"Changing 1 bit in a given message only alters 1 bit in its hash."**
  - ✅ **No**, due to the Avalanche Effect, flipping even 1 bit in the message cascades to flip approximately 50% of the bits everywhere throughout the SHA-256 fingerprint.
- ❌ **"Verifying the signature proves my computer hasn't been hacked."**
  - ✅ **No**, verification only proves that the mathematical signature was created by the corresponding private key on a specific block of bytes. If the sender's computer was compromised and the hacker signed malicious data, the math will still verify successfully!

## Section 16. Summary
To recap the lifecycle of a secure digital signature:

1. You write a **message**.
2. The system converts it into raw **bytes**.
3. The system computes a fixed-length **hash fingerprint** (SHA-256) of those bytes.
4. The system applies secure **padding** (RSA-PSS) adding randomness and structure.
5. The numerical form of this padded block is processed through the RSA **signing formula** $s = Block^d \pmod{n}$ using the **private key** and broadcast to the public.
6. A receiver grabs the mathematical signature and passes it back through the RSA **verification formula** $v = s^e \pmod{n}$ using the public **public key**.
7. The receiver algorithm ensures the isolated result geometrically and perfectly matches the expected structure of the given padded hash of the active message! If both ends align perfectly, authenticity and zero-tampering integrity are mathematically proven.

## Section 17. Practice exercises
Try running and modifying this notebook physically yourself to strengthen your understanding!
1. Run the `try...except` verification cell manually, swap out `real_message` for `hacked_message`, and watch it flag an `InvalidSignature` correctly.
2. Attempt generating a completely new *second* RSA key pair above, isolate its totally unique Public Key, and try to verify `secure_signature` against it instead. Notice it immediately triggers catastrophic mismatch failure.
3. Take `secure_signature` and manually slice the end bytes out (`corrupted = secure_signature[:-5]`). Does `.verify()` instantly snap it up?
4. Try defining your own local text file reading block (`with open('sample.txt', 'rb') as f: doc = f.read()`), generate matching signature outputs, and act them securely over.